# 🍕 Pizza Store Descriptive Statistics

**The Story:** You manage a pizza chain with 50 stores. Each store reports daily sales, average order value, customer ratings, and delivery times. Let's use descriptive statistics to understand the business.

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
n_stores = 50

df = pd.DataFrame({
    'store_id': [f'Store_{i+1}' for i in range(n_stores)],
    'city': np.random.choice(['New York', 'Chicago', 'LA', 'Houston', 'Phoenix'], n_stores),
    'daily_sales': np.round(np.random.normal(500, 120, n_stores)).astype(int),
    'avg_order_value': np.round(np.random.normal(18, 4, n_stores), 2),
    'customer_rating': np.round(np.random.uniform(2.5, 5.0, n_stores), 1),
    'delivery_time_min': np.round(np.random.exponential(25, n_stores) + 10, 1),
    'num_employees': np.random.randint(5, 25, n_stores),
})

# Add a couple of outlier stores
df.loc[0, 'daily_sales'] = 1200   # flagship store
df.loc[1, 'daily_sales'] = 80     # struggling store
df.loc[2, 'delivery_time_min'] = 95.0  # very slow store

print(f'Dataset: {len(df)} stores')
df.head(10)

## 1. The Big Picture — `.describe()`
One line gives you count, mean, std, min, quartiles, and max for every numeric column.

In [ ]:
df.describe().round(2)

## 2. Measures of Center — Mean, Median, Mode
**Question:** What does a *typical* store look like?

In [ ]:
from scipy import stats

center = pd.DataFrame({
    'Mean': df[['daily_sales', 'avg_order_value', 'delivery_time_min']].mean(),
    'Median': df[['daily_sales', 'avg_order_value', 'delivery_time_min']].median(),
    'Mode': df[['daily_sales', 'avg_order_value', 'delivery_time_min']].mode().iloc[0],
    'Mean - Median': df[['daily_sales', 'avg_order_value', 'delivery_time_min']].mean() - df[['daily_sales', 'avg_order_value', 'delivery_time_min']].median(),
}).round(2)

print('📊 Measures of Center')
print('=' * 60)
print(center)
print()
print('💡 When Mean > Median → data is right-skewed (a few high values pulling the mean up)')
print('💡 delivery_time_min has the biggest gap — some stores are VERY slow')

## 3. Measures of Spread — Std Dev, Variance, Range, IQR
**Question:** How *different* are the stores from each other?

In [ ]:
cols = ['daily_sales', 'avg_order_value', 'delivery_time_min']

spread = pd.DataFrame({
    'Std Dev': df[cols].std(),
    'Variance': df[cols].var(),
    'Min': df[cols].min(),
    'Max': df[cols].max(),
    'Range': df[cols].max() - df[cols].min(),
    'IQR (Q3-Q1)': df[cols].quantile(0.75) - df[cols].quantile(0.25),
}).round(2)

print('📏 Measures of Spread')
print('=' * 70)
print(spread)
print()
print('💡 High std dev = stores are very different from each other')
print('💡 IQR is more robust than Range (not affected by outliers)')

## 4. Visualize It — Histograms & Box Plots

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('🍕 Pizza Chain — Store Performance Overview', fontsize=14, fontweight='bold')

for i, col in enumerate(cols):
    # Histogram
    axes[0, i].hist(df[col], bins=15, color='#7c6aff', alpha=0.7, edgecolor='white')
    axes[0, i].axvline(df[col].mean(), color='#f45d6d', linestyle='--', label=f'Mean: {df[col].mean():.1f}')
    axes[0, i].axvline(df[col].median(), color='#22d3a7', linestyle='--', label=f'Median: {df[col].median():.1f}')
    axes[0, i].set_title(col)
    axes[0, i].legend(fontsize=8)

    # Box plot
    bp = axes[1, i].boxplot(df[col], vert=True, patch_artist=True,
                            boxprops=dict(facecolor='#7c6aff', alpha=0.5),
                            medianprops=dict(color='#22d3a7', linewidth=2),
                            flierprops=dict(marker='o', markerfacecolor='#f45d6d', markersize=8))
    axes[1, i].set_title(f'{col} (box plot)')

plt.tight_layout()
plt.show()

print('💡 Red dots in box plots = outliers (beyond 1.5 × IQR)')
print('💡 delivery_time_min is right-skewed — most stores are fast, but a few are very slow')

## 5. Quartiles, Percentiles & IQR — Explained on a Bell Curve

**The Story:** Imagine lining up all 50 pizza stores from worst to best daily sales. Quartiles split that line into **4 equal groups**:

- **Q1 (25th percentile):** 25% of stores sell LESS than this
- **Q2 (50th percentile):** The median — right in the middle
- **Q3 (75th percentile):** 75% of stores sell LESS than this
- **IQR = Q3 − Q1:** The range of the middle 50%. Used to detect outliers.

In [ ]:
from scipy.stats import norm

sales = df['daily_sales']
q1 = sales.quantile(0.25)
q2 = sales.quantile(0.50)  # median
q3 = sales.quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

print(f'📊 Daily Sales — Quartile Breakdown')
print(f'=' * 45)
print(f'  Min            : {sales.min()}')
print(f'  Q1  (25th %ile): {q1}')
print(f'  Q2  (Median)   : {q2}')
print(f'  Q3  (75th %ile): {q3}')
print(f'  Max            : {sales.max()}')
print(f'  IQR (Q3 − Q1)  : {iqr}')
print(f'  Lower fence    : {lower_fence:.0f}  (anything below = outlier)')
print(f'  Upper fence    : {upper_fence:.0f}  (anything above = outlier)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── LEFT: Bell curve with quartile zones ──
ax = axes[0]
mu, sigma = sales.mean(), sales.std()
x = np.linspace(mu - 4*sigma, mu + 4*sigma, 500)
y = norm.pdf(x, mu, sigma)

# Full curve outline
ax.plot(x, y, color='#7c6aff', linewidth=2)

# Shade the 4 quartile zones with different colors
colors = ['#f45d6d', '#f5b731', '#22d3a7', '#5eaeff']
labels = ['Bottom 25%\n(struggling)', 'Q1→Q2 (25-50%)\n(below average)', 'Q2→Q3 (50-75%)\n(above average)', 'Top 25%\n(star stores)']
bounds = [x.min(), q1, q2, q3, x.max()]

for i in range(4):
    mask = (x >= bounds[i]) & (x <= bounds[i+1])
    ax.fill_between(x[mask], y[mask], alpha=0.35, color=colors[i], label=labels[i])

# Vertical lines at Q1, Q2, Q3
for val, name, c in [(q1, f'Q1 = {q1:.0f}', '#f45d6d'), (q2, f'Q2 (Median) = {q2:.0f}', '#f5b731'), (q3, f'Q3 = {q3:.0f}', '#22d3a7')]:
    ax.axvline(val, color=c, linewidth=2, linestyle='--')
    ax.text(val, max(y)*1.02, name, ha='center', fontsize=9, fontweight='bold', color=c)

# IQR bracket
bracket_y = max(y) * 0.55
ax.annotate('', xy=(q3, bracket_y), xytext=(q1, bracket_y),
            arrowprops=dict(arrowstyle='<->', color='white', lw=2))
ax.text((q1+q3)/2, bracket_y + max(y)*0.04, f'IQR = {iqr:.0f}\n(middle 50%)',
        ha='center', fontsize=10, fontweight='bold', color='white',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#252840', edgecolor='#7c6aff', alpha=0.9))

# Outlier fences
for fence, label in [(lower_fence, f'Lower fence\n{lower_fence:.0f}'), (upper_fence, f'Upper fence\n{upper_fence:.0f}')]:
    if x.min() < fence < x.max():
        ax.axvline(fence, color='#e879a8', linewidth=1.5, linestyle=':')
        ax.text(fence, max(y)*0.15, label, ha='center', fontsize=7, color='#e879a8')

ax.set_title('🔔 Quartiles on the Bell Curve — daily_sales', fontsize=12, fontweight='bold')
ax.set_xlabel('Daily Sales', fontsize=10)
ax.set_ylabel('Density', fontsize=10)
ax.legend(loc='upper left', fontsize=7, framealpha=0.9)

# ── RIGHT: Annotated number line ──
ax2 = axes[1]
ax2.set_xlim(-0.5, 10.5)
ax2.set_ylim(-1, 7)
ax2.axis('off')
ax2.set_title('📏 Quartiles as a Number Line', fontsize=12, fontweight='bold')

# Main line
ax2.plot([0, 10], [3, 3], color='white', linewidth=3)

# Tick marks and labels
positions = {'Min': (0, sales.min()), 'Q1': (2.5, q1), 'Median': (5, q2), 'Q3': (7.5, q3), 'Max': (10, sales.max())}
tick_colors = {'Min': '#8892b0', 'Q1': '#f45d6d', 'Median': '#f5b731', 'Q3': '#22d3a7', 'Max': '#8892b0'}

for name, (pos, val) in positions.items():
    c = tick_colors[name]
    ax2.plot([pos, pos], [2.6, 3.4], color=c, linewidth=3)
    ax2.text(pos, 4.0, f'{name}', ha='center', fontsize=11, fontweight='bold', color=c)
    ax2.text(pos, 2.0, f'{val:.0f}', ha='center', fontsize=12, fontweight='bold', color='white')

# Shade quartile zones on the number line
zone_info = [
    (0, 2.5, '#f45d6d', '25%'),
    (2.5, 5, '#f5b731', '25%'),
    (5, 7.5, '#22d3a7', '25%'),
    (7.5, 10, '#5eaeff', '25%'),
]
for x0, x1, c, pct in zone_info:
    ax2.fill_between([x0, x1], [2.7, 2.7], [3.3, 3.3], color=c, alpha=0.3)
    ax2.text((x0+x1)/2, 3.0, pct, ha='center', va='center', fontsize=9, fontweight='bold', color=c)

# IQR bracket
ax2.annotate('', xy=(7.5, 5.2), xytext=(2.5, 5.2),
             arrowprops=dict(arrowstyle='<->', color='white', lw=2))
ax2.text(5, 5.6, f'IQR = {iqr:.0f}  (middle 50% of stores)',
         ha='center', fontsize=10, fontweight='bold', color='white',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#252840', edgecolor='#7c6aff'))

# Outlier explanation
ax2.text(5, 0.5, f'⚠️ Outlier if below {lower_fence:.0f} or above {upper_fence:.0f}  (Q1 − 1.5×IQR  or  Q3 + 1.5×IQR)',
         ha='center', fontsize=9, color='#e879a8', style='italic')

plt.tight_layout()
plt.show()

print(f'💡 Q1={q1:.0f}: 25% of stores sell less than this — these need attention')
print(f'💡 Q2={q2:.0f}: The median — half the stores are above, half below')
print(f'💡 Q3={q3:.0f}: 75% of stores sell less than this — stores above Q3 are your stars')
print(f'💡 IQR={iqr:.0f}: The "normal" range. Anything outside the fences ({lower_fence:.0f}–{upper_fence:.0f}) is suspicious')

## 6. Outlier Detection — IQR Method
**Question:** Which stores are *weirdly* different?

In [ ]:
def find_outliers_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return series[(series < lower) | (series > upper)]

print('🚨 Outlier Stores')
print('=' * 50)
for col in cols:
    outliers = find_outliers_iqr(df[col])
    if len(outliers) > 0:
        print(f'\n{col}: {len(outliers)} outlier(s)')
        for idx in outliers.index:
            print(f'  → {df.loc[idx, "store_id"]} ({df.loc[idx, "city"]}): {outliers[idx]}')
    else:
        print(f'\n{col}: No outliers ✅')

print('\n💡 Outliers are not always bad — Store_1 might be a flagship, Store_2 might need help')

## 7. Group By City — Compare Performance
**Question:** Which city's stores are doing best?

In [ ]:
city_stats = df.groupby('city').agg(
    stores=('store_id', 'count'),
    avg_daily_sales=('daily_sales', 'mean'),
    median_daily_sales=('daily_sales', 'median'),
    std_daily_sales=('daily_sales', 'std'),
    avg_rating=('customer_rating', 'mean'),
    avg_delivery=('delivery_time_min', 'mean'),
).round(1).sort_values('avg_daily_sales', ascending=False)

print('🏙️ Performance by City')
print('=' * 80)
print(city_stats)
print()
print('💡 Look at std_daily_sales — high std means inconsistent stores in that city')

## 8. Skewness & Kurtosis
**Question:** Is the data bell-shaped or lopsided?

In [ ]:
from scipy.stats import norm

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('📐 Skewness & Kurtosis — Is the data bell-shaped or lopsided?', fontsize=14, fontweight='bold')

for i, col in enumerate(cols):
    skew_val = df[col].skew()
    kurt_val = df[col].kurtosis()
    label = 'Right-skewed' if skew_val > 0.5 else 'Left-skewed' if skew_val < -0.5 else 'Roughly symmetric'
    color = '#f45d6d' if abs(skew_val) > 0.5 else '#22d3a7'

    # Row 1: Histogram with normal curve overlay
    ax = axes[0, i]
    n, bins, patches = ax.hist(df[col], bins=15, density=True, color='#7c6aff', alpha=0.6, edgecolor='white')
    # Overlay a normal curve to show how far from bell-shaped it is
    x_fit = np.linspace(df[col].min(), df[col].max(), 200)
    y_fit = norm.pdf(x_fit, df[col].mean(), df[col].std())
    ax.plot(x_fit, y_fit, color='#f5b731', linewidth=2, linestyle='--', label='Perfect bell curve')
    ax.axvline(df[col].mean(), color='#f45d6d', linestyle='--', alpha=0.8, label=f'Mean: {df[col].mean():.1f}')
    ax.axvline(df[col].median(), color='#22d3a7', linestyle='--', alpha=0.8, label=f'Median: {df[col].median():.1f}')
    ax.set_title(f'{col}\nSkew={skew_val:.2f} ({label})', fontsize=10, color=color)
    ax.legend(fontsize=7)

    # Row 2: Annotated bar showing skewness & kurtosis values
    ax2 = axes[1, i]
    bars = ax2.barh(['Skewness', 'Kurtosis'], [skew_val, kurt_val],
                    color=['#7c6aff', '#e879a8'], alpha=0.7, edgecolor='white')
    ax2.axvline(0, color='#22d3a7', linewidth=2, linestyle='-', label='Ideal (0 = bell curve)')
    for bar, val in zip(bars, [skew_val, kurt_val]):
        ax2.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
                 f'{val:.2f}', va='center', fontsize=11, fontweight='bold')
    ax2.set_title(f'{col} — Shape Metrics', fontsize=10)
    ax2.set_xlim(min(-1.5, skew_val - 0.5), max(3, kurt_val + 1))
    ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()

print('💡 Yellow dashed line = what a perfect bell curve would look like')
print('💡 When the histogram doesn\'t match the yellow line → data is skewed')
print('💡 Skewness > 0 → tail stretches RIGHT (like delivery times — most are fast, a few are very slow)')
print('💡 Kurtosis > 0 → heavier tails than a bell curve (more extreme values than expected)')

## 9. Correlation — Quick Peek
**Question:** Do any of these metrics move together?

In [ ]:
import seaborn as sns

numeric_cols = ['daily_sales', 'avg_order_value', 'customer_rating', 'delivery_time_min', 'num_employees']
corr = df[numeric_cols].corr().round(2)

# --- Heatmap ---
fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)  # hide upper triangle (redundant)
cmap = sns.diverging_palette(10, 150, as_cmap=True)   # red ↔ green
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap=cmap, center=0,
            vmin=-1, vmax=1, square=True, linewidths=1, linecolor='#2d3148',
            cbar_kws={'label': 'Correlation (r)', 'shrink': 0.8},
            annot_kws={'size': 12, 'fontweight': 'bold'}, ax=ax)
ax.set_title('🔗 Correlation Heatmap — Which metrics move together?', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

print('💡 Green = positive correlation (both go up together)')
print('💡 Red = negative correlation (one goes up, other goes down)')
print('💡 White/pale = no relationship')
print()

# --- Scatter plots for top correlations ---
# Find the strongest off-diagonal correlations
pairs = []
for i_idx in range(len(numeric_cols)):
    for j_idx in range(i_idx+1, len(numeric_cols)):
        pairs.append((numeric_cols[i_idx], numeric_cols[j_idx], abs(corr.iloc[i_idx, j_idx])))
pairs.sort(key=lambda x: x[2], reverse=True)
top_pairs = pairs[:3]  # top 3 strongest

fig2, axes2 = plt.subplots(1, 3, figsize=(16, 4.5))
fig2.suptitle('🔍 Top 3 Strongest Correlations — Scatter Plots', fontsize=13, fontweight='bold')

for idx, (col_x, col_y, r_val) in enumerate(top_pairs):
    ax = axes2[idx]
    ax.scatter(df[col_x], df[col_y], c='#7c6aff', alpha=0.6, s=40, edgecolors='white', linewidth=0.5)
    # Trend line
    z = np.polyfit(df[col_x], df[col_y], 1)
    x_line = np.linspace(df[col_x].min(), df[col_x].max(), 100)
    ax.plot(x_line, np.polyval(z, x_line), color='#f45d6d', linewidth=2, linestyle='--')
    actual_r = corr.loc[col_x, col_y]
    color = '#22d3a7' if abs(actual_r) > 0.3 else '#f5b731'
    ax.set_title(f'r = {actual_r:.2f}', fontsize=12, color=color, fontweight='bold')
    ax.set_xlabel(col_x, fontsize=9)
    ax.set_ylabel(col_y, fontsize=9)

plt.tight_layout()
plt.show()

print('💡 Tight cluster along the line = strong correlation')
print('💡 Random cloud = no correlation')
print('💡 Remember: correlation ≠ causation!')

## 📝 Summary

| What we learned | Tool used | Key finding |
|---|---|---|
| What's a typical store? | Mean, Median | ~500 daily sales, $18 avg order |
| How different are stores? | Std Dev, IQR | Sales vary by ~$120, delivery times vary a lot |
| Any weird stores? | IQR outlier detection | Flagship store (high sales), one very slow store |
| Which city is best? | GroupBy + Mean | Compare across cities |
| Is data bell-shaped? | Skewness | Delivery times are right-skewed |
| Do metrics relate? | Correlation | Quick check for relationships |